# Banco LLM | Laboratorio 3. RAG y herramientas del copiloto de Marketing
Proyecto Integrador · Grupo 1

### Flujo y funcionamiento del copiloto de Marketing
Este laboratorio propone un sistema que ayuda al equipo de marketing de un banco ficticio llamado Banco LLM. El sistema utiliza un agente de IA para preparar correos promocionales personalizados. A partir del identificador interno del cliente (client_id) y la fecha de campaña, el agente consulta la API de perfiles para conocer sus preferencias y verificar sus permisos de contacto. Luego consulta la API de campañas para identificar promociones vigentes que se ajusten a sus intereses y cumplan los criterios de elegibilidad.

Con la información de las promociones y las directrices de comunicación del banco, recuperadas mediante RAG, el agente redacta tres alternativas de correo. El equipo de marketing elige una y puede editarla. A partir de la versión elegida, el agente consulta la guía visual y genera un prompt que describe una imagen acorde con la promoción y los atributos autorizados del cliente. Finalmente, el equipo guarda el correo como borrador o aprueba su entrega en una bandeja de prueba.

## 1. Conocimiento, chunking y metadatos

### ¿Qué estrategia de chunking elegimos y por qué?

Dividimos por estructura. Cada norma del manual ocupa una página y cada directriz visual empieza con un encabezado de nivel dos. Conservamos el título junto al contenido y usamos solapamiento cero. Así, una cifra como el límite del asunto permanece junto a la regla que indica qué caracteres contar. La portada y el índice del manual quedan fuera del corpus.

### ¿Qué metadatos usamos para filtrar y cuáles enriquecen el contexto?

| Campo de ChromaDB | Contenido | Uso en el agente |
|---|---|---|
| `id` | Identificador estable del fragmento, como `VIS-01` | Cita y actualización del fragmento. |
| `documents` | Título y texto completo de la norma o directriz | Evidencia que recibe el modelo para responder. |
| `metadatas.categoria` | `comunicacion` o `visual` | Filtro exacto en `where` para acotar la búsqueda antes de recuperar los vecinos. |
| Otros metadatos | Fuente, título, tipo documental, número de chunk, versión y página para el PDF | Contexto de la cita y trazabilidad hasta el original. |
| `embeddings` | Vector de `text-embedding-3-small` para el texto | Selección por similitud coseno. |

La distancia se calcula en cada consulta. Cuanto menor es, más cercano resulta el fragmento. La respuesta también debe respaldarse en su contenido.

In [1]:
import hashlib
import json
import os
import re
import unicodedata
from datetime import date, datetime, timezone
from pathlib import Path
from typing import Literal
from uuid import uuid4

import chromadb
import httpx
import numpy as np
import pandas as pd
from agent_framework import Agent, InMemoryHistoryProvider, Message, SlidingWindowStrategy, apply_compaction, tool
from agent_framework.openai import OpenAIChatClient, OpenAIEmbeddingClient
from azure.core.exceptions import ClientAuthenticationError
from azure.identity import DefaultAzureCredential
from chromadb.config import Settings
from dotenv import load_dotenv
from IPython.display import display
from pydantic import BaseModel, Field
from pypdf import PdfReader

BASE = Path.cwd()

In [2]:
def cargar_chunks():
    manual = BASE / "conocimiento/manual_comunicacion.pdf"
    guia = BASE / "conocimiento/guia_visual.md"
    chunks = []
    for pagina, hoja in enumerate(PdfReader(manual).pages[2:], 3):
        contenido = hoja.extract_text().split("MKT-MAN-03   /", 1)[0]
        norma = re.search(
            r"^NORMA (\d+)\s*\n(.*?)\nCriterio institucional\n(.*)", contenido, re.M | re.S
        )
        if norma is None:
            raise ValueError(f"Sección desconocida en la página {pagina}.")
        numero, titulo, texto = norma.groups()
        titulo = " ".join(titulo.split())
        chunks.append(
            {
                "id": f"MAN-{numero}",
                "text": titulo + "\nCriterio institucional\n" + texto.strip(),
                "metadata": {
                    "fuente": manual.name,
                    "titulo": titulo,
                    "categoria": "comunicacion",
                    "tipo_documento": "manual",
                    "numero_chunk": int(numero),
                    "pagina": pagina,
                    "version": "1.0",
                },
            }
        )
    for seccion in re.split(r"^## ", guia.read_text(encoding="utf-8"), flags=re.M)[1:]:
        cabecera, texto = seccion.split("\n", 1)
        ident, titulo = cabecera.split(" | ", 1)
        chunks.append(
            {
                "id": ident,
                "text": titulo + "\n" + texto.strip(),
                "metadata": {
                    "fuente": guia.name,
                    "titulo": titulo,
                    "categoria": "visual",
                    "tipo_documento": "guia",
                    "numero_chunk": int(ident.split("-")[1]),
                    "version": "1.0",
                },
            }
        )
    return chunks

In [3]:
chunks = cargar_chunks()
assert len(chunks) == len({c["id"] for c in chunks}) == 30
display(pd.DataFrame([c["metadata"] for c in chunks]).groupby(["fuente","categoria"]).size().to_frame("chunks"))
print(chunks[4]["id"], chunks[4]["metadata"], "\n", chunks[4]["text"])
assert all(c["text"].strip() and c["metadata"]["fuente"] for c in chunks)
assert {c["metadata"]["categoria"] for c in chunks} == {"comunicacion", "visual"}

,,chunks
fuente,categoria,
guia_visual.md,visual,7
manual_comunicacion.pdf,comunicacion,23


MAN-05 {'fuente': 'manual_comunicacion.pdf', 'titulo': 'Asunto y preencabezado', 'categoria': 'comunicacion', 'tipo_documento': 'manual', 'numero_chunk': 5, 'pagina': 7, 'version': '1.0'} 
 Asunto y preencabezado
Criterio institucional
El asunto comercial admite hasta 55 caracteres, incluidos espacios. El preencabezado
admite hasta 90 caracteres, incluidos espacios. Cada uno debe aportar información
distinta.
Aplicación en el correo
Abre con el producto o la ocasión. Si el porcentaje depende de un máximo, el cuerpo
debe indicar el tope con claridad. Evita urgencia artificial y promesas incondicionales.
Caso de Marketing
Marketing presenta una campaña gastronómica. La primera versión repite el nombre del
comercio en ambos campos y deja la mecánica para el final.
Redacción de referencia
Asunto: Una invitación para compartir pastas. Preencabezado: Conoce los platos
incluidos y cómo se aplica el beneficio.
Control de aprobación
Cuenta letras, espacios y signos antes de publicar. Si no cabe

## 2. Conexión e indexación
Microsoft Entra ID, gpt-5.6-luna, text-embedding-3-small y ChromaDB local. La huella del corpus permite reutilizar una colección íntegra y completar cargas interrumpidas.

In [4]:
class Indice:
    def __init__(self, cliente, endpoint, deployment, directorio=None):
        self.cliente = cliente
        self.endpoint = endpoint
        self.deployment = deployment
        self.directorio = directorio or BASE / "chroma_proyecto"
        self.coleccion = None

    async def vectorizar(self, textos):
        respuesta = await self.cliente.get_embeddings(textos)
        vectores = np.asarray([r.vector for r in respuesta], dtype=np.float32)
        if (
            vectores.ndim != 2
            or len(vectores) != len(textos)
            or not np.isfinite(vectores).all()
            or not (np.linalg.norm(vectores, axis=1) > 0).all()
        ):
            raise ValueError("Embeddings inválidos o incompletos.")
        return vectores

    async def preparar(self):
        chunks = cargar_chunks()
        muestra = await self.vectorizar(["Directrices comerciales sintéticas."])
        firma = json.dumps(
            [self.endpoint, self.deployment, muestra.shape[1], chunks],
            ensure_ascii=False,
            sort_keys=True,
        )
        nombre = "banco_llm_" + hashlib.sha256(firma.encode()).hexdigest()[:20]
        db = chromadb.PersistentClient(
            path=str(self.directorio), settings=Settings(anonymized_telemetry=False)
        )
        coleccion = db.get_or_create_collection(
            nombre, embedding_function=None, configuration={"hnsw": {"space": "cosine"}}
        )
        existentes = set(coleccion.get(include=[])["ids"])
        faltantes = [c for c in chunks if c["id"] not in existentes]
        for i in range(0, len(faltantes), 12):
            lote = faltantes[i : i + 12]
            textos = [c["text"] for c in lote]
            vectores = await self.vectorizar(textos)
            if vectores.shape[1] != muestra.shape[1]:
                raise ValueError("Dimensión inconsistente en el lote.")
            coleccion.upsert(
                ids=[c["id"] for c in lote],
                documents=textos,
                metadatas=[c["metadata"] for c in lote],
                embeddings=vectores.tolist(),
            )
        if set(coleccion.get(include=[])["ids"]) != {c["id"] for c in chunks}:
            raise ValueError("Los identificadores del índice difieren del corpus.")
        self.coleccion = coleccion
        return {
            "coleccion": nombre,
            "chunks": coleccion.count(),
            "dimension": muestra.shape[1],
            "nuevos": len(faltantes),
        }

    async def buscar(self, pregunta, categoria=None, top_k=4):
        if self.coleccion is None:
            raise RuntimeError("Primero prepara el índice.")
        if not 1 <= top_k <= 10:
            raise ValueError("top_k debe estar entre 1 y 10.")
        filtros = {"where": {"categoria": categoria}} if categoria else {}
        total = len(self.coleccion.get(include=[], **filtros)["ids"])
        if not total:
            return []
        vector = await self.vectorizar([pregunta])
        hits = self.coleccion.query(
            query_embeddings=vector.tolist(),
            include=["documents", "metadatas", "distances"],
            n_results=min(top_k, total),
            **filtros,
        )
        return [
            {"id": i, "text": t, "metadata": m, "distancia": float(d)}
            for i, t, m, d in zip(
                *(hits[k][0] for k in ("ids", "documents", "metadatas", "distances")), strict=True
            )
        ]

In [5]:
def clientes_foundry():
    archivo = next((BASE / n for n in (".env", "env") if (BASE / n).is_file()), None)
    if archivo:
        load_dotenv(archivo, override=False)
    endpoint = os.getenv("FOUNDRY_MODEL_ENDPOINT", "").strip().rstrip("/")
    modelo = os.getenv("FOUNDRY_MODEL_DEPLOYMENT", "").strip()
    embedding = os.getenv("FOUNDRY_EMBEDDING_DEPLOYMENT", "text-embedding-3-small").strip()
    if not endpoint or (modelo, embedding) != ("gpt-5.6-luna", "text-embedding-3-small"):
        raise ValueError("Configura el endpoint y los dos deployments del laboratorio en env o .env.")
    credencial = DefaultAzureCredential()
    chat = OpenAIChatClient(model=modelo, base_url=endpoint, credential=credencial)
    emb = OpenAIEmbeddingClient(model=embedding, base_url=endpoint, credential=credencial)
    return chat, Indice(emb, endpoint, embedding)

In [6]:
from azure.core.exceptions import ClientAuthenticationError
from openai import AuthenticationError, PermissionDeniedError

def error_acceso(exc):
    vistos = set()
    while exc is not None and id(exc) not in vistos:
        vistos.add(id(exc))
        if isinstance(exc, (ClientAuthenticationError, AuthenticationError, PermissionDeniedError)):
            return type(exc).__name__
        exc = exc.__cause__ or exc.__context__
    return None

async def ejecutar_foundry(nombre, operacion):
    try:
        return await operacion
    except Exception as exc:
        motivo = error_acceso(exc)
        if motivo is None:
            raise
        bloqueos[nombre] = f"Bloqueada por acceso a Foundry ({motivo})"
        print(nombre, "·", bloqueos[nombre])
        return None

evidencias = []
bloqueos = {}
cliente, indice = clientes_foundry()
datos_indice = await ejecutar_foundry("Embeddings e índice", indice.preparar())
conectado = datos_indice is not None
estado = bloqueos.get("Embeddings e índice", "Foundry disponible.")
if conectado:
    almacenado = indice.coleccion.get(ids=["VIS-01"], include=["documents", "metadatas", "embeddings"])
    assert almacenado["ids"] == ["VIS-01"] and almacenado["documents"][0]
    assert almacenado["metadatas"][0]["categoria"] == "visual"
    assert len(almacenado["embeddings"][0]) == datos_indice["dimension"]
    print(datos_indice)
    print("VIS-01 ·", almacenado["metadatas"][0]["fuente"], "· dimensión", len(almacenado["embeddings"][0]))

{'coleccion': 'banco_llm_f184aaeebd7c5d919c39', 'chunks': 30, 'dimension': 1536, 'nuevos': 0}
VIS-01 · guia_visual.md · dimensión 1536


## 3. Cada componente por separado, sin agente
### 3.1 API de perfiles

In [7]:
URL_PERFILES = "http://127.0.0.1:8001"
URL_CAMPANAS = "http://127.0.0.1:8002"


class APIIndisponible(RuntimeError):
    pass


async def solicitar(metodo, url, comando, **kwargs):
    try:
        async with httpx.AsyncClient(timeout=10.0, trust_env=False) as cliente:
            respuesta = await cliente.request(metodo, url, **kwargs)
    except httpx.RequestError as exc:
        raise APIIndisponible(
            f"No se obtuvo respuesta de {url}. Comprueba la terminal de {comando}."
        ) from exc
    if respuesta.status_code in (404, 422):
        raise ValueError(str(respuesta.json()["detail"]))
    respuesta.raise_for_status()
    return respuesta.json()


async def consultar_perfil(client_id):
    if not re.fullmatch(r"CLI-\d{4}", client_id):
        raise ValueError("Identificador de cliente inválido.")
    return await solicitar("GET", f"{URL_PERFILES}/perfiles/{client_id}", "python api_perfiles.py")

In [8]:
perfil = await consultar_perfil("CLI-1001")
assert perfil["client_id"] == "CLI-1001" and len(perfil["afinidades"]) == 3
assert all(len(a["subcategorias"]) == 2 for a in perfil["afinidades"])
display(pd.DataFrame(perfil["afinidades"]))
evidencias.append({"prueba":"Perfil directo","ok":True,"salida":perfil})

,prioridad,categoria,subcategorias
0,1,Viajes,"[Hoteles, Vuelos internacionales]"
1,2,Tecnología,"[Computadoras, Smartphones]"
2,3,Vehículos,"[Mantenimiento, Lavado]"


### 3.2 API de campañas

In [9]:
async def consultar_campanas(mes, anio, afinidades):
    return await solicitar(
        "POST", f"{URL_CAMPANAS}/campanas", "python api_campanas.py",
        json={"mes": mes, "anio": anio, "afinidades": afinidades},
    )

In [10]:
entrada_campanas = [{"categoria":"Viajes","subcategorias":["Hoteles"]}]
fichas = await consultar_campanas(11, 2026, entrada_campanas)
assert [f["id"] for f in fichas] == ["LLM-33"] and "20" in fichas[0]["Beneficio"]
display(pd.DataFrame(fichas)[["id","Comercio","Beneficio","Mecánica y restricciones","Inicio","Fin"]])
evidencias.append({"prueba":"Campañas directas","ok":True,"salida":fichas})

,id,Comercio,Beneficio,Mecánica y restricciones,Inicio,Fin
0,LLM-33,Posada Demo Lumbre,20 % de descuento,Alojamiento estándar desde S/ 300. Ahorro máxi...,2026-11-01,2026-11-30


### 3.3 RAG documental | Misma pregunta con y sin filtro

In [11]:
consulta = "¿Qué relación de aspecto tiene la cabecera del correo y qué tercio de la imagen debe quedar libre?"
if conectado:
    comparacion = []
    for categoria in (None, "visual"):
        hits = await indice.buscar(consulta, categoria)
        assert hits and all(categoria is None or h["metadata"]["categoria"] == categoria for h in hits)
        comparacion.extend({"filtro":categoria or "sin filtro", "id":h["id"],"fuente":h["metadata"]["fuente"],
                            "fragmento": h["text"][:180], "distancia":h["distancia"]} for h in hits)
    ok = any(h["id"] == "VIS-01" for h in hits)
    display(pd.DataFrame(comparacion))
    print("¿Se recuperó la regla de encuadre con filtro?", ok)
    evidencias.append({"prueba":"RAG directo","ok":ok,"comparacion":comparacion})
else:
    bloqueos["RAG directo"] = estado
    print(estado)

,filtro,id,fuente,fragmento,distancia
0,sin filtro,VIS-01,guia_visual.md,Encuadre para correo\nEl formato horizontal de...,0.290004
1,sin filtro,MAN-06,manual_comunicacion.pdf,Estructura del correo\nCriterio institucional\...,0.507565
2,sin filtro,MAN-05,manual_comunicacion.pdf,Asunto y preencabezado\nCriterio institucional...,0.531929
3,sin filtro,VIS-07,guia_visual.md,Entrega del prompt seleccionado\nEl prompt vis...,0.534685
4,visual,VIS-01,guia_visual.md,Encuadre para correo\nEl formato horizontal de...,0.290004
5,visual,VIS-07,guia_visual.md,Entrega del prompt seleccionado\nEl prompt vis...,0.534685
6,visual,VIS-03,guia_visual.md,Viaje individual\nPara una campaña de pasajes ...,0.642621
7,visual,VIS-06,guia_visual.md,"Tecnología y movilidad\nPara audio, utiliza el...",0.667545


¿Se recuperó la regla de encuadre con filtro? True


## 4. Cada componente por separado, con agente
Cada prueba crea un agente de Microsoft Agent Framework con una sola función registrada y una sesión nueva. La traza proviene de las funciones ejecutadas. También se contrasta la respuesta final con el dato del caso.

### Memoria y formato de las propuestas

In [12]:
PERFIL = "consultar_perfil"
CAMPANAS = "consultar_campanas"
RAG = "buscar_directrices"
TODAS = (PERFIL, CAMPANAS, RAG)


class MemoriaCampanas(InMemoryHistoryProvider):
    def __init__(self):
        super().__init__("historial", load_messages=True)
        self.ventana = SlidingWindowStrategy(keep_last_groups=16)

    async def after_run(self, *, agent, session, context, state):
        if context.response and context.response.text:
            turno = [
                Message("user", [context.input_messages[-1].text]),
                Message("assistant", [context.response.text]),
            ]
            await self.save_messages(session.session_id, turno, state=state)
            historial = await self.get_messages(session.session_id, state=state)
            state["messages"] = await apply_compaction(historial, strategy=self.ventana)


class Propuesta(BaseModel):
    campana_id: str
    asunto: str = Field(min_length=1, max_length=55)
    apertura: str = Field(min_length=1, max_length=450)
    cta: str = Field(min_length=1, max_length=120)
    fuentes: list[str] = Field(min_length=1)


class TresPropuestas(BaseModel):
    propuestas: list[Propuesta] = Field(min_length=3, max_length=3)

### Afinidades y elegibilidad

In [13]:
def clave(valor):
    return "".join(
        c
        for c in unicodedata.normalize("NFKD", valor.strip().casefold())
        if not unicodedata.combining(c)
    )


class Afinidad(BaseModel):
    categoria: str
    subcategorias: list[str] = Field(min_length=1, max_length=8)


def elegibles(perfil, fichas, fecha):
    pares = {
        (clave(a["categoria"]), clave(s)): a["prioridad"]
        for a in perfil["afinidades"]
        for s in a["subcategorias"]
    }
    candidatos = {}
    for f in fichas:
        par = (clave(f["Categoría"]), clave(f["Subcategoría"]))
        if (
            par in pares
            and perfil["edad"] >= 18
            and date.fromisoformat(f["Inicio"]) <= fecha <= date.fromisoformat(f["Fin"])
            and perfil["segmento"] in f["Segmentos elegibles"].split(", ")
            and set(perfil["productos"]) & set(f["Productos elegibles"].split(", "))
        ):
            candidatos[f["id"]] = (pares[par], f)
    return [f for _, f in sorted(candidatos.values(), key=lambda x: (x[0], x[1]["id"]))]


def datos_publicables(perfil):
    return {
        "client_id": perfil["client_id"],
        "nombre_de_pila": perfil["nombre_completo"].split()[0],
        "segmento": perfil["segmento"],
        "productos": perfil["productos"],
        "afinidades": perfil["afinidades"],
    }


def pares(afinidades):
    return {(clave(a["categoria"]), clave(s)) for a in afinidades for s in a["subcategorias"]}

### Registro de herramientas y sesión del agente

In [14]:
class Ejecucion:
    def __init__(self, cliente, indice, componentes=TODAS, client_id=None, fecha=None):
        self.indice = indice
        self.client_id = client_id
        self.fecha = fecha
        self.traza = []
        self.error_api = None
        self.perfil = None
        self.fichas = {}
        self.consultadas = set()
        self.fragmentos = {}
        self.componentes = tuple(componentes)
        self.herramientas = self.registrar()
        self.agente = Agent(
            client=cliente,
            name="CopilotoCampanas",
            instructions=(BASE / "instrucciones_agente.md").read_text(encoding="utf-8"),
            tools=self.herramientas,
            default_options={"store": False},
            context_providers=[MemoriaCampanas()],
        )
        self.sesion = self.agente.create_session()

    def registrar(self):
        @tool(
            name=PERFIL,
            description="Consulta la API local por client_id. Devuelve el perfil comercial permitido y las tres afinidades con sus subcategorías.",
        )
        async def perfil(client_id: str) -> dict:
            if self.client_id and self.client_id != client_id:
                raise ValueError("El cliente no corresponde al encargo.")
            try:
                p = await consultar_perfil(client_id)
            except APIIndisponible as exc:
                self.error_api = exc
                raise
            self.perfil = p
            self.traza.append(
                {"tool": PERFIL, "client_id": client_id, "afinidades": len(p["afinidades"])}
            )
            if not p["permisos"]["personalizacion_texto"] or not p["permisos"]["contacto_correo"]:
                return {"client_id": client_id, "estado": "sin_autorizacion"}
            return datos_publicables(p)

        @tool(
            name=CAMPANAS,
            description="Busca en la API mensual por mes, año y lista de afinidades, cada una con categoria y subcategorias. En un encargo envía las tres afinidades del perfil juntas.",
        )
        async def campanas(mes: int, anio: int, afinidades: list[Afinidad]) -> list[dict]:
            filtros = [
                a.model_dump() if isinstance(a, Afinidad) else Afinidad.model_validate(a).model_dump()
                for a in afinidades
            ]
            if self.client_id:
                if not self.perfil or not all(
                    self.perfil["permisos"][k] for k in ("contacto_correo", "personalizacion_texto")
                ):
                    raise ValueError("Primero consulta el perfil y verifica los permisos.")
                if (mes, anio) != (self.fecha.month, self.fecha.year) or not pares(filtros) <= pares(
                    self.perfil["afinidades"]
                ):
                    raise ValueError("Utiliza el periodo y las afinidades del encargo.")
            try:
                filas = await consultar_campanas(mes, anio, filtros)
            except APIIndisponible as exc:
                self.error_api = exc
                raise
            if self.client_id:
                filas = elegibles(self.perfil, filas, self.fecha)
            self.consultadas.update(pares(filtros))
            self.fichas.update({f["id"]: f for f in filas})
            self.traza.append(
                {
                    "tool": CAMPANAS,
                    "mes": mes,
                    "anio": anio,
                    "afinidades": filtros,
                    "ids": [f["id"] for f in filas],
                }
            )
            return filas

        @tool(
            name=RAG,
            description="Recupera fragmentos de ChromaDB con su ID, texto, archivo, página y distancia coseno. La categoría opcional es comunicacion o visual.",
        )
        async def rag(
            pregunta: str, categoria: Literal["comunicacion", "visual"] | None = None
        ) -> list[dict]:
            if self.indice is None:
                raise RuntimeError("Índice no disponible.")
            hits = await self.indice.buscar(pregunta, categoria)
            self.fragmentos.update({h["id"]: h for h in hits})
            self.traza.append(
                {
                    "tool": RAG,
                    "pregunta": pregunta,
                    "categoria": categoria,
                    "ids": [h["id"] for h in hits],
                    "distancias": [h["distancia"] for h in hits],
                }
            )
            return hits

        registro = {PERFIL: perfil, CAMPANAS: campanas, RAG: rag}
        return [registro[n] for n in self.componentes]

    async def preguntar(self, pregunta):
        self.error_api = None
        try:
            respuesta = await self.agente.run(pregunta, session=self.sesion)
        finally:
            if self.error_api is not None:
                raise self.error_api
        return respuesta.text

In [15]:
async def probar_aislado(componente, pregunta, validar):
    ejecucion = Ejecucion(cliente, indice, (componente,))
    respuesta = await ejecutar_foundry("Agente + " + componente, ejecucion.preguntar(pregunta))
    if respuesta is None:
        return
    llamadas = [t["tool"] for t in ejecucion.traza]
    ok = bool(llamadas) and set(llamadas) == {componente} and validar(respuesta, ejecucion)
    print(respuesta)
    display(pd.DataFrame(ejecucion.traza))
    print("Verificado" if ok else "Revisar respuesta o llamadas")
    evidencias.append({"prueba":"Agente + " + componente,"ok":ok,"pregunta":pregunta,"respuesta":respuesta,"traza":ejecucion.traza})

### 4.1 Agente con la herramienta de perfiles

In [16]:
await probar_aislado(PERFIL, "Consulta el perfil CLI-1001. Indica su segmento y enumera sus tres categorías principales.",
                     lambda r,e: all(x in r.casefold() for x in ("preferente","viajes","tecnología","vehículos")))

CAMPAÑA

Segmento: **Preferente**

Tres categorías principales:
1. **Viajes**
2. **Tecnología**
3. **Vehículos**


,tool,client_id,afinidades
0,consultar_perfil,CLI-1001,3


Verificado


### 4.2 Agente con la herramienta de campañas

In [17]:
await probar_aislado(CAMPANAS, "Consulta noviembre de 2026, categoría Viajes, subcategoría Hoteles. Devuelve el ID, el comercio y el beneficio publicado.",
                     lambda r,e: "LLM-33" in r and "posada demo lumbre" in r.casefold() and bool(re.search(r"20\s*%", r)))

CAMPAÑA

- **ID:** LLM-33  
- **Comercio:** Posada Demo Lumbre  
- **Beneficio publicado:** 20 % de descuento


,tool,mes,anio,afinidades,ids
0,consultar_campanas,11,2026,"[{'categoria': 'Viajes', 'subcategorias': ['Ho...",[LLM-33]


Verificado


### 4.3 Agente con el RAG

In [18]:
if conectado or ("Agente + " + PERFIL in bloqueos and "Agente + " + CAMPANAS in bloqueos):
    await probar_aislado(RAG, consulta + " Consulta la categoría visual y cita el fragmento y el archivo.",
                         lambda r,e: "VIS-01" in e.fragmentos and "[VIS-01]" in r and "guia_visual.md" in r
                         and "derech" in r.casefold() and bool(re.search(r"3\s*[:x×/]\s*2", r)))
else:
    bloqueos["Agente + " + RAG] = estado
    print(estado)

DIRECTRIZ

La cabecera del correo debe tener una relación de aspecto horizontal de **3:2**. El **tercio derecho** de la imagen debe quedar libre para la composición posterior. [VIS-01, **guia_visual.md**]


,tool,pregunta,categoria,ids,distancias
0,buscar_directrices,¿Qué relación de aspecto tiene la cabecera del...,visual,"[VIS-01, VIS-07, VIS-03, VIS-06]","[0.29000377655029297, 0.5346848964691162, 0.64..."


Revisar respuesta o llamadas


## 5. Un agente con los tres componentes
El encargo parte de un client_id y una fecha. El agente consulta las afinidades y las campañas, recupera las reglas editoriales y genera tres alternativas. El sistema comprueba los permisos, la cobertura de las seis subcategorías y la elegibilidad antes de exponer el resultado.

In [19]:
def validar_redaccion(asunto, apertura, cta):
    texto = " ".join([asunto, apertura, cta])
    if re.search(r"\d|%|S/|gratis|gratuit|sin intereses|garantiz|sin costo|https?://", texto, re.I):
        raise ValueError("Reserva las condiciones comerciales y las cifras para el bloque de la ficha.")


def armar_correo(p, perfil, ficha):
    validar_redaccion(p.asunto, p.apertura, p.cta)
    cuerpo = (
        f"Hola, {perfil['nombre_completo'].split()[0]}.\n\n{p.apertura}\n\n"
        f"{ficha['Comercio']} · {ficha['Beneficio']}. {ficha['Mecánica y restricciones']}\n"
        f"Vigencia: {ficha['Inicio']} al {ficha['Fin']}. {ficha['Canal y territorio']}\n"
        f"Productos participantes: {ficha['Productos elegibles']}. Segmentos: {ficha['Segmentos elegibles']}.\n"
        f"{ficha['Aplicaciones']} {ficha['Disponibilidad']}\n\n{p.cta}"
    )
    return {**p.model_dump(), "cuerpo": cuerpo}

In [20]:
async def preparar_encargo(cliente, indice, client_id, fecha):
    e = Ejecucion(cliente, indice, client_id=client_id, fecha=fecha)
    pregunta = (
        f"ENCARGO. Cliente {client_id}, fecha comercial {fecha.isoformat()}. Invoca consultar_perfil. "
        "Si tiene permisos, invoca consultar_campanas con todas sus afinidades y subcategorías para el mes y año indicados. "
        "Después invoca buscar_directrices con categoria comunicacion para la redacción. "
        "Redacta tres alternativas distintas para una o varias de las primeras fichas elegibles según prioridad. "
        "Devuelve exclusivamente JSON con propuestas, una lista de tres objetos con campana_id, asunto, apertura, cta y fuentes "
        "(lista de IDs documentales exactos). Asunto de hasta 55 caracteres. Apertura de 1 o 2 frases, un CTA. "
        "Estos tres textos son editoriales, sin números, porcentajes ni promesas comerciales. El sistema agrega los términos exactos. "
        "Si faltan permisos o campañas, explica el motivo y termina."
    )
    for intento in range(2):
        texto = await e.preguntar(pregunta)
        if e.perfil and not all(
            e.perfil["permisos"][k] for k in ("contacto_correo", "personalizacion_texto")
        ):
            raise ValueError(
                "El cliente no tiene ambos permisos de contacto y personalización de texto."
            )
        completo = e.perfil and e.consultadas == pares(e.perfil["afinidades"])
        if completo and not e.fichas:
            raise ValueError("No hay campañas elegibles para las afinidades y la fecha del encargo.")
        try:
            if not completo or not any(
                h["metadata"]["categoria"] == "comunicacion" for h in e.fragmentos.values()
            ):
                raise ValueError(
                    "Falta consultar el perfil, todas sus afinidades o las directrices de comunicación."
                )
            limpio = texto.strip().removeprefix(chr(96) * 3 + "json").removesuffix(chr(96) * 3).strip()
            propuestas = TresPropuestas.model_validate_json(limpio).propuestas
            if len({clave(p.asunto) for p in propuestas}) != 3:
                raise ValueError("Las tres alternativas deben tener asuntos distintos.")
            correos = []
            for p in propuestas:
                if p.campana_id not in e.fichas or not set(p.fuentes) <= set(e.fragmentos):
                    raise ValueError("La campaña o la cita no provienen de las herramientas invocadas.")
                correos.append(armar_correo(p, e.perfil, e.fichas[p.campana_id]))
            break
        except ValueError as exc:
            if intento:
                raise ValueError(f"No se logró validar el encargo: {exc}") from exc
            pregunta = f"Corrige tu entrega del mismo encargo. {str(exc)[:500]}. Completa las llamadas pendientes y devuelve el JSON solicitado."
    return {
        "client_id": client_id,
        "fecha": fecha.isoformat(),
        "perfil": datos_publicables(e.perfil),
        "fichas": e.fichas,
        "fuentes": {k: v["metadata"] for k, v in e.fragmentos.items()},
        "correos": correos,
        "traza": e.traza,
    }

In [21]:
CLIENT_ID = "CLI-1001"
FECHA_DEMO = date(2026, 11, 22)
encargo = None
if conectado or ("Agente + " + PERFIL in bloqueos and "Agente + " + CAMPANAS in bloqueos):
    encargo = await ejecutar_foundry(
        "Agente con los tres componentes", preparar_encargo(cliente, indice, CLIENT_ID, FECHA_DEMO)
    )
else:
    bloqueos["Agente con los tres componentes"] = estado
if encargo is not None:
    assert {t["tool"] for t in encargo["traza"]} == {PERFIL,CAMPANAS,RAG}
    assert len(encargo["correos"]) == 3
    display(pd.DataFrame(encargo["traza"]))
    for i,p in enumerate(encargo["correos"], 1):
        print(i, p["asunto"], "\n", p["cuerpo"], "\n", p["fuentes"], "\n")
    evidencias.append({"prueba":"Agente con los tres componentes","ok":True,"encargo":encargo})
else:
    print(bloqueos["Agente con los tres componentes"])

,tool,client_id,afinidades,mes,anio,ids,pregunta,categoria,distancias
0,consultar_perfil,CLI-1001,3,NaN,NaN,NaN,NaN,NaN,NaN
1,consultar_campanas,NaN,"[{'categoria': 'Viajes', 'subcategorias': ['Ho...",11.0,2026.0,"[LLM-33, LLM-34]",NaN,NaN,NaN
2,buscar_directrices,NaN,NaN,NaN,NaN,"[MAN-17, MAN-06, MAN-05, MAN-03]",Redacción de correos promocionales de campañas...,comunicacion,"[0.28251975774765015, 0.3148658275604248, 0.34..."


1 Una pausa especial en Posada Demo Lumbre 
 Hola, Alma.

Conoce la propuesta de Posada Demo Lumbre y revisa cómo aplica el beneficio de la campaña.

Posada Demo Lumbre · 20 % de descuento. Alojamiento estándar desde S/ 300. Ahorro máximo S/ 100. Alimentación y traslados excluidos.
Vigencia: 2026-11-01 al 2026-11-30. pago presencial en Perú.
Productos participantes: Crédito Horizonte, Crédito Cumbre. Segmentos: Preferente, Patrimonial.
una por cliente adulto y campaña. No acumulable. cupo total sintético de 100 beneficios por campaña, sujeto a disponibilidad.

Consulta las condiciones de la campaña. 
 ['MAN-03', 'MAN-05', 'MAN-06', 'MAN-17'] 

2 Tu próxima estadía empieza aquí 
 Hola, Alma.

Posada Demo Lumbre te invita a descubrir una propuesta pensada para tus próximos planes de viaje.

Posada Demo Lumbre · 20 % de descuento. Alojamiento estándar desde S/ 300. Ahorro máximo S/ 100. Alimentación y traslados excluidos.
Vigencia: 2026-11-01 al 2026-11-30. pago presencial en Perú.
Produc

## 6. Flujo completo de Marketing
### Selección, edición y prompt visual
La alternativa 2 está preseleccionada para la presentación. El cuerpo conserva el bloque comercial de la ficha. Al confirmar la elección, el agente recupera las reglas visuales para la versión editada.

In [22]:
async def generar_visual(cliente, indice, encargo, numero, asunto, apertura, cta):
    if numero not in (1, 2, 3):
        raise ValueError("Elige una alternativa del 1 al 3.")
    perfil = await consultar_perfil(encargo["client_id"])
    p = Propuesta.model_validate(
        {**encargo["correos"][numero - 1], "asunto": asunto, "apertura": apertura, "cta": cta}
    )
    ficha = encargo["fichas"][p.campana_id]
    if not all(perfil["permisos"][k] for k in ("contacto_correo", "personalizacion_texto")):
        raise ValueError("El perfil no autoriza este encargo.")
    correo = armar_correo(p, perfil, ficha)
    permisos = perfil["permisos"]
    atributos = {
        k: perfil[k]
        for k in permisos["atributos_visuales"]
        if permisos["personalizacion_visual"]
        and k in ("edad", "genero", "estado_civil", "hijos")
        and perfil[k] not in (None, "sin dato")
    }
    e = Ejecucion(cliente, indice, (RAG,))
    entrada = {
        "alternativa": numero,
        "asunto": asunto,
        "apertura": apertura,
        "cta": cta,
        "ficha": ficha,
        "atributos_visuales_autorizados": atributos,
    }
    texto = await e.preguntar(
        "PROMPT VISUAL. Marketing ha seleccionado y editado el correo descrito en el JSON. "
        "Invoca buscar_directrices con categoria visual. Genera el prompt fotográfico en español para la versión elegida. "
        "Usa solamente atributos autorizados; si la lista está vacía, compón una escena de producto sin personas. "
        "Incluye personajes o producto, acción, encuadre, iluminación y exclusiones; sin nombres o IDs de clientes. "
        "Devuelve solo la descripción visual, sin encabezados ni referencias, que se registran aparte.\n"
        + json.dumps(entrada, ensure_ascii=False)
    )
    if not texto.strip() or not any(t["categoria"] == "visual" and t["ids"] for t in e.traza):
        raise ValueError("La etapa visual requiere un prompt y evidencia de la guía visual.")
    return {
        "numero": numero,
        "correo": correo,
        "prompt_visual": texto.strip(),
        "fuentes_visuales": list(e.fragmentos),
        "traza_visual": e.traza,
    }

In [23]:
ALTERNATIVA = 2
if encargo is not None:
    propuesta = encargo["correos"][ALTERNATIVA-1]
    seleccion = await generar_visual(cliente, indice, encargo, ALTERNATIVA, propuesta["asunto"],
                                     propuesta["apertura"], "Consulta los detalles de esta campaña.")
    print(seleccion["correo"]["cuerpo"], "\n\nPrompt visual\n", seleccion["prompt_visual"])
    display(pd.DataFrame(seleccion["traza_visual"]))
else:
    bloqueos["Flujo completo"] = bloqueos["Agente con los tres componentes"]
    print(bloqueos["Flujo completo"])

Hola, Alma.

Posada Demo Lumbre te invita a descubrir una propuesta pensada para tus próximos planes de viaje.

Posada Demo Lumbre · 20 % de descuento. Alojamiento estándar desde S/ 300. Ahorro máximo S/ 100. Alimentación y traslados excluidos.
Vigencia: 2026-11-01 al 2026-11-30. pago presencial en Perú.
Productos participantes: Crédito Horizonte, Crédito Cumbre. Segmentos: Preferente, Patrimonial.
una por cliente adulto y campaña. No acumulable. cupo total sintético de 100 beneficios por campaña, sujeto a disponibilidad.

Consulta los detalles de esta campaña. 

Prompt visual
 Fotografía editorial horizontal 3:2 de una mujer de 34 años viajando sola, en el vestíbulo elegante y acogedor de un hotel ficticio, junto a una maleta de mano, consultando tranquilamente una reserva en su teléfono mientras se dispone a registrarse. Vestimenta casual refinada y apropiada para un viaje, expresión relajada, apariencia natural y no identificable. Encuadre a media distancia, acción principal ubicada

,tool,pregunta,categoria,ids,distancias
0,buscar_directrices,Directrices visuales para crear el prompt foto...,visual,"[VIS-07, VIS-01, VIS-03, VIS-06]","[0.32634222507476807, 0.367160439491272, 0.463..."


### Borrador o entrega aprobada
La versión elegida se guarda en un JSON. Con `APROBAR_ENVIO = True`, se registra en la bandeja sintética del cliente.

In [24]:
async def guardar_correo(encargo, seleccion, aprobar, directorio):
    client_id = encargo["client_id"]
    if aprobar:
        perfil = await consultar_perfil(client_id)
        permisos = perfil["permisos"]
        if not (permisos["contacto_correo"] and permisos["personalizacion_texto"]):
            raise ValueError("El perfil no autoriza el envío.")
    estado = "entregado" if aprobar else "borrador"
    carpeta = directorio / ("bandeja" if aprobar else "borradores") / client_id
    carpeta.mkdir(parents=True, exist_ok=True)
    registro = {"client_id": client_id, "fecha_campana": encargo["fecha"], "estado": estado, **seleccion}
    contenido = json.dumps(registro, ensure_ascii=False, indent=2)
    firma = hashlib.sha256(contenido.encode()).hexdigest()[:16]
    ruta = carpeta / f"{encargo['fecha']}_{firma}.json"
    temporal = carpeta / f"{uuid4().hex}.tmp"
    try:
        temporal.write_text(contenido, encoding="utf-8")
        temporal.replace(ruta)
    finally:
        temporal.unlink(missing_ok=True)
    return ruta

In [25]:
APROBAR_ENVIO = False
if encargo is not None:
    ruta = await guardar_correo(encargo, seleccion, APROBAR_ENVIO, BASE / "resultados")
    registro = json.loads(ruta.read_text(encoding="utf-8"))
    assert registro["correo"] == seleccion["correo"]
    print(registro["estado"], ruta.relative_to(BASE))
    evidencias.append({"prueba": "Flujo completo", "ok": True, "archivo": str(ruta.relative_to(BASE))})
else:
    bloqueos["Flujo completo"] = bloqueos["Agente con los tres componentes"]
    print(bloqueos["Flujo completo"])

borrador resultados\borradores\CLI-1001\2026-11-22_1bfc116d63a26e90.json


## 7. Evaluación documental del RAG
Cinco consultas independientes. Cada caso registra pregunta, filtro, fuentes, distancias, respuesta y justificación. Cada consulta utiliza el agente con la herramienta RAG.

In [26]:
casos = [
    {"caso": "Directa", "pregunta": "¿Cuántos caracteres, incluidos los espacios, admite como máximo el asunto comercial?",
     "filtro": None, "objetivo": "MAN-05", "patrones": [r"\b55\b", r"espacio"]},
    {"caso": "Reformulada", "pregunta": "Al redactar la línea que verá primero el destinatario en su bandeja, ¿cuál es el límite contando también los blancos?",
     "filtro": None, "objetivo": "MAN-05", "patrones": [r"\b55\b", r"espacio|blanco"]},
    {"caso": "Otro chunk", "pregunta": "¿Quién aprueba la versión antes del envío y qué cambios obligan a cotejarla otra vez con la ficha comercial?",
     "filtro": None, "objetivo": "MAN-20", "patrones": [r"marketing",r"importe|monto",r"comercio|establecimiento",r"plazo"]},
    {"caso": "Con filtro", "pregunta": "¿Qué relación de aspecto tiene la cabecera del correo y qué tercio queda libre?",
     "filtro": "visual", "objetivo": "VIS-01", "patrones": [r"3\s*[:x×/]\s*2|tres a dos",r"derech"]},
    {"caso": "Información ausente", "pregunta": "¿Cuántas millas aéreas de cortesía recibe un cliente por cada hora de retraso de su vuelo? Indica la cantidad exacta.",
     "filtro": None, "objetivo": None, "patrones": []},
]
display(pd.DataFrame(casos)[["caso","pregunta","filtro","objetivo"]].fillna("Ninguno"))

,caso,pregunta,filtro,objetivo
0,Directa,"¿Cuántos caracteres, incluidos los espacios, a...",Ninguno,MAN-05
1,Reformulada,Al redactar la línea que verá primero el desti...,Ninguno,MAN-05
2,Otro chunk,¿Quién aprueba la versión antes del envío y qu...,Ninguno,MAN-20
3,Con filtro,¿Qué relación de aspecto tiene la cabecera del...,visual,VIS-01
4,Información ausente,¿Cuántas millas aéreas de cortesía recibe un c...,Ninguno,Ninguno


In [27]:
def normalizar(texto):
    return "".join(c for c in unicodedata.normalize("NFKD", texto.casefold()) if not unicodedata.combining(c))


def evaluar(caso, salida):
    respuesta = salida["respuesta"].strip()
    texto = normalizar(respuesta)
    ids = {h["id"] for h in salida["fragmentos"]}
    citas = set(re.findall(r"\[((?:MAN|VIS)-\d{2})\]", respuesta))
    if not citas.issubset(ids):
        return "incorrecto", "Cita un fragmento que no se recuperó."
    if not caso["objetivo"]:
        cuerpo = re.sub(r"\[(?:man|vis)-\d+\]", "", texto)
        cuantifica = re.search(r"\b(?:\d+(?:[.,]\d+)?|cero|una|dos|tres|cinco|diez|veinte|cien|mil)\s+millas?\b", cuerpo)
        if texto.startswith("informacion insuficiente") and "milla" in texto and not cuantifica:
            return "correcto", "Reconoce la ausencia y no atribuye una cantidad de millas."
        return "incorrecto", "Falta la abstención explícita o se atribuye un beneficio sin respaldo."
    if caso["objetivo"] not in ids:
        return "incorrecto", "El fragmento de referencia no llegó al contexto."
    if caso["filtro"] and any(h["metadata"]["categoria"] != caso["filtro"] for h in salida["fragmentos"]):
        return "incorrecto", "Hay fragmentos fuera de la categoría solicitada."
    fuente = next(h["metadata"]["fuente"] for h in salida["fragmentos"] if h["id"] == caso["objetivo"])
    hechos = [bool(re.search(p, texto)) for p in caso["patrones"]]
    if all(hechos) and caso["objetivo"] in citas and fuente in respuesta:
        return "correcto", "Coinciden los datos esperados, el fragmento citado y el archivo fuente."
    if any(hechos):
        return "parcial", f"Coinciden {sum(hechos)}/{len(hechos)} criterios de contenido. Revisar la respuesta y su cita."
    return "incorrecto", "El fragmento llegó al contexto, pero faltan los datos solicitados."


resultados = []
if conectado:
    for caso in casos:
        e = Ejecucion(cliente, indice, (RAG,))
        indicacion = caso["pregunta"] + (" Usa la herramienta con filtro visual." if caso["filtro"] else " Usa la herramienta sin filtro.")
        respuesta = await ejecutar_foundry("RAG documental · " + caso["caso"], e.preguntar(indicacion))
        if respuesta is None:
            continue
        salida = {"pregunta":caso["pregunta"],"filtro":caso["filtro"],"fragmentos":list(e.fragmentos.values()),
                  "respuesta":respuesta,"traza":e.traza}
        evaluacion, motivo = evaluar(caso, salida)
        if not e.traza or any(t["categoria"] != caso["filtro"] for t in e.traza):
            evaluacion, motivo = "incorrecto", "No se invocó el RAG con el filtro del caso."
        resultados.append({"caso": caso["caso"], **salida, "evaluacion": evaluacion,
                           "justificacion": motivo, "objetivo": caso["objetivo"]})
        print(f"\n{caso['caso']} | {evaluacion}\n{caso['pregunta']}\n{salida['respuesta']}\n{motivo}")
    if resultados:
        display(pd.DataFrame([{"caso": r["caso"], "filtro": r["filtro"] or "sin filtro",
                           "fuentes y chunks": ", ".join(f"{h['id']} ({h['metadata']['fuente']}, {h['distancia']:.3f})" for h in r["fragmentos"]),
                           "evaluación": r["evaluacion"]} for r in resultados]))
else:
    print(estado)


Directa | parcial
¿Cuántos caracteres, incluidos los espacios, admite como máximo el asunto comercial?
DIRECTRIZ

El asunto comercial admite como máximo **55 caracteres, incluidos los espacios**. [MAN-05, manual_comunicacion.pdf]
Coinciden 2/2 criterios de contenido. Revisar la respuesta y su cita.

Reformulada | parcial
Al redactar la línea que verá primero el destinatario en su bandeja, ¿cuál es el límite contando también los blancos?
DIRECTRIZ

El **preencabezado**, la línea visible en la bandeja de entrada, admite hasta **90 caracteres, incluidos los espacios en blanco**. [MAN-05, manual_comunicacion.pdf]
Coinciden 1/2 criterios de contenido. Revisar la respuesta y su cita.

Otro chunk | parcial
¿Quién aprueba la versión antes del envío y qué cambios obligan a cotejarla otra vez con la ficha comercial?
DIRECTRIZ

Marketing selecciona una de las tres propuestas, revisa sus condiciones y aprueba la versión antes del envío. Debe cotejarse nuevamente con la ficha comercial cualquier c

,caso,filtro,fuentes y chunks,evaluación
0,Directa,sin filtro,"MAN-05 (manual_comunicacion.pdf, 0.315), MAN-0...",parcial
1,Reformulada,sin filtro,"MAN-05 (manual_comunicacion.pdf, 0.484), VIS-0...",parcial
2,Otro chunk,sin filtro,"MAN-20 (manual_comunicacion.pdf, 0.363), MAN-1...",parcial
3,Con filtro,visual,"VIS-01 (guia_visual.md, 0.294), VIS-07 (guia_v...",parcial
4,Información ausente,sin filtro,"MAN-12 (manual_comunicacion.pdf, 0.594), MAN-2...",correcto


### ¿En qué preguntas funcionó bien el RAG y qué dificultades se observaron?
La reflexión se calcula con las respuestas y las fuentes efectivamente recuperadas durante esta ejecución.
La pregunta sobre millas exige abstención. Un fragmento puede mencionar viajes y recuperar una distancia baja sin contener una política sobre retrasos. En la pregunta reformulada se contrasta si el mismo fragmento respalda la respuesta pese al cambio de vocabulario.

In [28]:
if resultados:
    aciertos = [r["caso"] for r in resultados if r["evaluacion"] == "correcto"]
    print("El agente cumplió los criterios en:", ", ".join(aciertos) or "ninguna consulta")
    for r in resultados:
        print(f"{r['caso']} · {r['evaluacion']}. {r['justificacion']}")
    for r in resultados[:2]:
        posiciones = [i for i,h in enumerate(r["fragmentos"],1) if h["id"] == r["objetivo"]]
        print(r["caso"], "· posición de MAN-05:", posiciones[0] if posiciones else "fuera de los resultados")
    print("Abstención ante información ausente:", resultados[-1]["evaluacion"])
    destino = BASE / "resultados"
    destino.mkdir(exist_ok=True)
    evidencia = destino / f"laboratorio_{datetime.now(timezone.utc):%Y%m%dT%H%M%S%fZ}.json"
    evidencia.write_text(json.dumps({"integracion":evidencias,"documentales":resultados}, ensure_ascii=False,indent=2),encoding="utf-8")
    print(evidencia.name)
else:
    print("Aún no se puede atribuir un acierto o un fallo semántico al RAG. "
          "No hubo respuestas evaluables para estas cinco preguntas. Consulta el diagnóstico en el resumen.")

El agente cumplió los criterios en: Información ausente
Directa · parcial. Coinciden 2/2 criterios de contenido. Revisar la respuesta y su cita.
Reformulada · parcial. Coinciden 1/2 criterios de contenido. Revisar la respuesta y su cita.
Otro chunk · parcial. Coinciden 4/4 criterios de contenido. Revisar la respuesta y su cita.
Con filtro · parcial. Coinciden 2/2 criterios de contenido. Revisar la respuesta y su cita.
Información ausente · correcto. Reconoce la ausencia y no atribuye una cantidad de millas.
Directa · posición de MAN-05: 1
Reformulada · posición de MAN-05: 1
Abstención ante información ausente: correcto
laboratorio_20261123T105342397079Z.json


## 8. Diseño de herramientas
### A. Tabla de Tools propuestas

| Nombre | Propósito y momento de uso | Entradas | Salidas | Fuente | Justificación |
|---|---|---|---|---|---|
| `consultar_perfil` | Iniciar el encargo con las afinidades del cliente. | `client_id` interno | Perfil comercial autorizado con tres categorías priorizadas y dos subcategorías por categoría, o estado `sin_autorizacion`. | API de perfiles, puerto 8001. Un JSON sintético por cliente. | El identificador no permite inferir preferencias ni permisos. |
| `consultar_campanas` | Buscar ofertas del periodo para las afinidades recuperadas. | Mes, año y lista de categorías con sus subcategorías. | Fichas con ID, comercio, beneficio, vigencia y condiciones. Durante el encargo se filtran por elegibilidad. | API de campañas, puerto 8002. Catálogo mensual sintético. | Los importes, topes y fechas deben coincidir con la ficha vigente. |
| `buscar_directrices` | Consultar las normas al redactar los correos y la guía visual después de la selección. | Pregunta y categoría documental opcional, `comunicacion` o `visual`. | Fragmentos con ID, texto, fuente, metadatos y distancia. | Índice ChromaDB del manual institucional y la guía visual. | Marketing necesita identificar la norma que respalda cada decisión editorial. |

### B. Evaluación de Agent as a Tool

¿Necesitamos que el agente principal invoque a otro agente como herramienta? Para este flujo, un único agente es suficiente. Las dos APIs resuelven consultas estructuradas y el RAG aporta las normas. Marketing elige y edita una propuesta. Con esa versión y las reglas visuales, el agente redacta el prompt de imagen. La selección y la aprobación final corresponden al operador.

Si Marketing requiriera una revisión editorial independiente, el agente principal podría delegarla en un revisor. Recibiría el correo seleccionado, la ficha comercial y los fragmentos citados. Devolvería discrepancias, referencias y una recomendación de revisión antes de la aprobación humana.

## 9. Resumen de pruebas

In [29]:
verificadas = {e["prueba"]: e["ok"] for e in evidencias}
ensayos = ["Embeddings e índice", "Perfil directo", "Campañas directas", "RAG directo",
           "Agente + " + PERFIL, "Agente + " + CAMPANAS, "Agente + " + RAG,
           "Agente con los tres componentes", "Flujo completo"]
if conectado:
    verificadas["Embeddings e índice"] = True
filas = []
for ensayo in ensayos:
    resultado = bloqueos.get(ensayo) or {True: "Verificada", False: "Revisar"}.get(verificadas.get(ensayo), "Pendiente")
    filas.append({"Prueba": ensayo, "Resultado": resultado})
por_caso = {r["caso"]: r for r in resultados}
for caso in casos:
    nombre = caso["caso"]
    resultado = por_caso[nombre]["evaluacion"] if nombre in por_caso else bloqueos.get("RAG documental · " + nombre, bloqueos.get("Embeddings e índice", "Pendiente"))
    filas.append({"Prueba": "RAG documental · " + nombre, "Resultado": resultado})
display(pd.DataFrame(filas))

,Prueba,Resultado
0,Embeddings e índice,Verificada
1,Perfil directo,Verificada
2,Campañas directas,Verificada
3,RAG directo,Verificada
4,Agente + consultar_perfil,Verificada
5,Agente + consultar_campanas,Verificada
6,Agente + buscar_directrices,Revisar
7,Agente con los tres componentes,Verificada
8,Flujo completo,Verificada
9,RAG documental · Directa,parcial


### Observación
Las pruebas verificaron la generación de embeddings, la indexación en ChromaDB y el uso conjunto de las tres herramientas. En las cuatro consultas documentales con respuesta disponible, el RAG recuperó el fragmento esperado, pero las respuestas del agente cumplieron parcialmente los criterios de contenido o citación. La prueba aislada de buscar_directrices quedó marcada para revisión. Ante la consulta sobre un beneficio ausente de la documentación, el agente reconoció que la información era insuficiente.

### Conclusión
El agente completó el flujo de Marketing al consultar el perfil del cliente, identificar campañas elegibles y utilizar las directrices institucionales para generar tres propuestas de correo. La principal mejora pendiente es lograr que cada respuesta documental incluya todos los datos solicitados y cite de forma consistente el fragmento y el archivo de origen.